In [9]:
import geopandas as gpd
import pandas as pd

# ==============================
# 1. Load datasets
# ==============================
fars = gpd.read_file("GA_supplemental/FARSmatched_GA.csv") # from Renee's road segmenting script
traffic = pd.read_csv("GA_supplemental/GDOT_Traffic_Counts_(AADT_and_Truck_Percent)_2008_to_2017.csv")

print(traffic.columns)

# ==============================
# 2. Convert traffic data to GeoDataFrame
# ==============================

fars = gpd.GeoDataFrame(
    fars,
    geometry=gpd.points_from_xy(fars['LONGITUD'], fars['LATITUDE']),
    crs="EPSG:4326"  # WGS84 geographic coordinates
)

traffic = gpd.GeoDataFrame(
    traffic,
    geometry=gpd.points_from_xy(traffic['Long'], traffic['Lat']),
    crs="EPSG:4326"
)

# ==============================
# 3. Project both to same projected CRS (for Georgia, EPSG:26917)
# ==============================
fars = fars.to_crs(26917)
traffic = traffic.to_crs(26917)

# ==============================
# 4. Spatially join: match fatal crashes to nearest traffic station
# ==============================
matched = gpd.sjoin_nearest(
    fars,
    traffic[["Station_ID", "geometry"]],
    how="left",
    max_distance=40,  # meters
    distance_col="dist"
)

# ==============================
# 5. Count number of crashes per station
# ==============================
crash_counts = (
    matched.groupby("Station_ID")
    .size()
    .reset_index(name="fatal_crash_count")
)

# ==============================
# 6. Merge crash counts back with traffic data
# ==============================
merged = traffic.merge(crash_counts, on="Station_ID", how="left")
merged["fatal_crash_count"] = merged["fatal_crash_count"].fillna(0)

# ==============================
# 7. Compute crash rate (done per 100m vehicle miles for ex)
# ==============================
# 2017 AADT is latest available
merged["MVMT"] = merged["AADT_2017"] * 365 / 1_000_000  # million vehicle miles/year approx
merged["crash_rate_per_100M_VMT"] = (
    merged["fatal_crash_count"] / (merged["MVMT"] + 1e-6) * 100
)

# ==============================
# 8. Save results
# ==============================
merged.to_csv("output/GA_fatal_crash_rates.csv", index=False)

print("Georgia crash rate file created successfully!")


Index(['Station_ID', 'Functional_Class', 'LatLong', 'Lat', 'Long', 'AADT_2017',
       'TruckPct_2017', 'AADT_2016', 'TruckPct_2016', 'AADT_2015',
       'TruckPct_2015', 'AADT_2014', 'TruckPct_2014', 'AADT_2013',
       'TruckPct_2013', 'AADT_2012', 'TruckPct_2012', 'AADT_2011',
       'TruckPct_2011', 'AADT_2010', 'TruckPct_2010', 'AADT_2009',
       'TruckPct_2009', 'AADT_2008', 'TruckPct_2008', 'ObjectId'],
      dtype='object')
Georgia crash rate file created successfully!


In [10]:
# ==============================
# 9. Check how many segments have non-zero crash rates
# ==============================
nonzero_count = (merged["crash_rate_per_100M_VMT"] > 0).sum()
total_segments = len(merged)
print(f"Non-zero crash rates: {nonzero_count} out of {total_segments} road segments")


Non-zero crash rates: 174 out of 26927 road segments


In [11]:
nonzero = merged[merged["crash_rate_per_100M_VMT"] > 0]
nonzero.to_csv("GA_supplemental/GA_fatal_crash_rates_nonzero.csv", index=False)
print("done!")

done!
